In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [2]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.ustable_cp import UStableConformalPredictor

Load data

In [3]:
input_points, output_points = load_data("friedman1")

In [4]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)

Instantiate predictor

In [5]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [6]:
predictor = KernelRegression(
    solver="lbfgs", loss_name=loss_name, loss_params=loss_params, lam=0.5
)

Instantiate region predictor

In [7]:
conformal_predictor = UStableConformalPredictor(
    predictor, non_conformity_name="absolute"
)
region_predictor = conformal_predictor.fit_predict(
    train_input_points, train_output_points, test_input_points
)

In [8]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

In [10]:
prediction_regions

[{'upper': [array([-2.15623391]),array([2.29205081])],
  'lower': [array([-0.01780943]),array([0.15362633])]},
 {'upper': [array([-2.12520785]),array([2.26901025])],
  'lower': [array([-0.04412848]),array([0.18793089])]},
 {'upper': [array([-2.0701114]),array([2.03362518])],
  'lower': [array([-0.27465727]),array([0.23817104])]},
 {'upper': [array([-1.66392904]),array([2.87607318])],
  'lower': [array([0.5638427]),array([0.64830144])]},
 {'upper': [array([-1.84630902]),array([2.65470226])],
  'lower': [array([0.3553035]),array([0.45308974])]},
 {'upper': [array([-2.48338407]),array([1.87607099])],
  'lower': [array([-0.43053592]),array([-0.17677715])]},
 {'upper': [array([-1.49646925]),array([2.78262212])],
  'lower': [array([0.46902285]),array([0.81713003])]},
 {'upper': [array([-2.59524546]),array([1.87886794])],
  'lower': [array([-0.42041223]),array([-0.29596529])]},
 {'upper': [array([-2.62879735]),array([1.77117693])],
  'lower': [array([-0.53932161]),array([-0.3182988])]},
 {'up

In [10]:
coverage_upper = np.mean(
    [
        test_output_point in prediction_region["upper"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_upper)

test coverage:  1.0


In [13]:
coverage_lower = np.mean(
    [
        test_output_point in prediction_region["lower"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_lower)

test coverage:  0.088
